In [1]:
from benchmark_pipeline.path import DATA
from benchmark_pipeline.config import (
    FEATURES,
    CITY_BBOXES,
    CITY_ECRS,
)
from benchmark_pipeline.generator.features import *

from benchmark_pipeline.loader.osm_loaders import load_tables
from benchmark_pipeline.loader.data_cleaning.name_matching import name_matching_cleaning
from benchmark_pipeline.generator.composite.generator_qcomposite import make_question_composite
from benchmark_pipeline.generator.composite.format_samples import format_composit_samples
from benchmark_pipeline.generator.register import register, BenchmarkConfig
from benchmark_pipeline.generator.multihop.generator_multihop import make_multihop_dataset

from benchmark_pipeline.generator.targets.mask_anchors import add_anchors_mask
from benchmark_pipeline.generator.targets.ellypses import add_ellypse
from benchmark_pipeline.generator.targets.anchors import add_anchors
from benchmark_pipeline.dataviz_tools.question_viz import montrer_questions

In [2]:
import time

OVERWRITE = False   # True regenerates the cities already done, False skips them

for city_name, bbox in CITY_BBOXES.items():
    city_start = time.perf_counter()
    print(f"\n=== [{list(CITY_BBOXES).index(city_name) + 1}/{len(CITY_BBOXES)}] {city_name} ===")
    # A city is done once the last file of the loop, written by make_multihop_dataset, exists:
    # a city stopped halfway (folder without multihop/) is generated again
    if not OVERWRITE and (DATA / city_name / "multihop" / "df_multihop.parquet").exists():
        print("    déjà générée, ignorée (OVERWRITE = False)")
        continue
    conf = BenchmarkConfig(name=city_name, bbox=bbox, ecrs=CITY_ECRS[city_name], rows_target=[])

    print("1/5 chargement OSM...")
    step_start = time.perf_counter()
    df_osm, df_street, df_area = load_tables(conf.bbox, conf.ecrs, city_name=conf.name)
    print(f"    {len(df_osm)} POIs, {len(df_street)} rues, {len(df_area)} zones ({time.perf_counter() - step_start:.0f} s)")

    print("2/5 dédoublonnage des noms...")
    df_all, (df_osm_clean, df_street_clean, df_area_clean) = name_matching_cleaning(df_osm, df_street, df_area)
    print(f"    gardés : {len(df_osm_clean)} POIs, {len(df_street_clean)} rues, {len(df_area_clean)} zones")

    print("3/5 génération des questions...")
    step_start = time.perf_counter()
    df_question = make_question_composite(
        df_osm_clean, 
        df_area_clean, 
        df_street_clean,
        nb_q_by_temp=conf.nb_q_by_temp, 
        nb_q_by_feat=conf.nb_q_by_feat,
        ratio_features=conf.ratio_features, 
        part_avec_features=conf.part_with_features,
        feature_cols=FEATURES, 
        seed=conf.seed, 
        is_paraphrase=conf.is_paraphrase
    )
    print(f"    {len(df_question)} questions ({time.perf_counter() - step_start:.0f} s)")

    print("4/5 enregistrement du benchmark...")
    step_start = time.perf_counter()
    register(
        DATA / conf.name, 
        df_question, 
        df_osm_clean, 
        df_street_clean, 
        df_area_clean, 
        df_all,
        conf, 
        format_composit_samples, 
        overwritte=True
    )
    print(f"    écrit dans {DATA / conf.name} ({time.perf_counter() - step_start:.0f} s)")

    print("5/5 questions multihop...")
    step_start = time.perf_counter()
    make_multihop_dataset(
        conf.nb_seed_questions, 
        conf.name, 
        conf.multihop_template,
        df_question, 
        df_osm_clean, 
        df_street_clean, 
        df_area_clean, 
        seed=conf.seed,
        nb_q_by_feat=conf.nb_q_by_feat, 
        ratio_features=conf.ratio_features,
        part_with_features=conf.part_with_features
    )
    print(f"    fait ({time.perf_counter() - step_start:.0f} s)")
    print(f"=== {city_name} terminé en {(time.perf_counter() - city_start) / 60:.1f} min ===")



=== [1/15] Paris ===
    déjà générée, ignorée (OVERWRITE = False)

=== [2/15] New York ===
    déjà générée, ignorée (OVERWRITE = False)

=== [3/15] Tokyo ===
    déjà générée, ignorée (OVERWRITE = False)

=== [4/15] Rio de Janeiro ===
    déjà générée, ignorée (OVERWRITE = False)

=== [5/15] Sydney ===
    déjà générée, ignorée (OVERWRITE = False)

=== [6/15] Cape Town ===
1/5 chargement OSM...


/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


251 morceaux de rue recollés
    1657 POIs, 26704 rues, 1014 zones (13 s)
2/5 dédoublonnage des noms...
lignes supprimées par source : {}
    gardés : 1405 POIs, 21638 rues, 1014 zones
3/5 génération des questions...
    26513 questions (41 s)
4/5 enregistrement du benchmark...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Cape Town (13 s)
5/5 questions multihop...
    fait (170 s)
=== Cape Town terminé en 4.0 min ===

=== [7/15] Lyon ===
1/5 chargement OSM...


/Users/sese/Documents/code/benchmark_pipeline/.venv/lib/python3.14/site-packages/quackosm/pbf_file_reader.py:3787: UserWarning: Select clause contains more than 100 columns (found 386 columns). Query might fail with insufficient memory resources. Consider applying more restrictive OsmTagsFilter for parsing.
  warnings.warn(
/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid
/Users/sese/Documents/code/benchmark_pipeline/.venv/lib/python3.14/site-packages/quackosm/pbf_file_reader.py:3787: UserWarning: Select clause contains more than 100 columns (found 399 columns). Query might fail with insufficient memory resources. Consider applying more restrictive OsmTagsFilter for parsing.
  warnings.warn(


42 morceaux de rue recollés


/Users/sese/Documents/code/benchmark_pipeline/.venv/lib/python3.14/site-packages/quackosm/pbf_file_reader.py:3787: UserWarning: Select clause contains more than 100 columns (found 145 columns). Query might fail with insufficient memory resources. Consider applying more restrictive OsmTagsFilter for parsing.
  warnings.warn(


... (13 outputs left out)

/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


11 morceaux de rue recollés
    32 POIs, 933 rues, 21 zones (349 s)
2/5 dédoublonnage des noms...
lignes supprimées par source : {}
    gardés : 27 POIs, 920 rues, 21 zones
3/5 génération des questions...
    1115 questions (6 s)
4/5 enregistrement du benchmark...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Polokwane (0 s)
5/5 questions multihop...
    fait (83 s)
=== Polokwane terminé en 7.3 min ===

=== [15/15] Chiayi ===
1/5 chargement OSM...


/Users/sese/Documents/code/benchmark_pipeline/.venv/lib/python3.14/site-packages/quackosm/pbf_file_reader.py:3787: UserWarning: Select clause contains more than 100 columns (found 168 columns). Query might fail with insufficient memory resources. Consider applying more restrictive OsmTagsFilter for parsing.
  warnings.warn(
/Users/sese/Documents/code/benchmark_pipeline/benchmark_pipeline/loader/osm_extractor.py:156: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  pois["geometry"] = pois.geometry.centroid


498 morceaux de rue recollés
    677 POIs, 338 rues, 170 zones (307 s)
2/5 dédoublonnage des noms...
lignes supprimées par source : {'poi': 5, 'street': 2}
    gardés : 58 POIs, 286 rues, 2 zones
3/5 génération des questions...
    2203 questions (3 s)
4/5 enregistrement du benchmark...
    écrit dans /Users/sese/Documents/code/benchmark_pipeline/data/Chiayi (1 s)
5/5 questions multihop...
    fait (54 s)
=== Chiayi terminé en 6.1 min ===


In [3]:
"""    #data= make_paraphrase(df_question_test, nb_generated=10)

    is_gseek_training = False
    if is_gseek_training:
        #sentence_model = SentenceEncoder()
        #df_question = add_anchors_mask(df_question, sentence_model)
        df_question = add_ellypse(df_question)
        df_question = add_anchors(df_question)
    #df_question = df_question.rename(columns={"pois": "results_poi_id"})
    # """

'    #data= make_paraphrase(df_question_test, nb_generated=10)\n\n    is_gseek_training = False\n    if is_gseek_training:\n        #sentence_model = SentenceEncoder()\n        #df_question = add_anchors_mask(df_question, sentence_model)\n        df_question = add_ellypse(df_question)\n        df_question = add_anchors(df_question)\n    #df_question = df_question.rename(columns={"pois": "results_poi_id"})\n    # '

In [4]:
#montrer_questions(df_question, df_osm, par_type=True)